# Qwen3-14B — reasoning on

Local run with `enable_thinking=True`. The answer is parsed from the completion, so this configuration is behavioural-only. `max_gen_tokens=2048`; shorter budgets truncate mid-reasoning and fail `parse_rate`.

A100 40GB. Typical wall time 3–6 hours.


## 1. Clone


In [ ]:
import os, sys, json, pathlib, subprocess

REPO_URL = "https://github.com/ryanmcdonough/behaviour-microscope.git"
REPO_DIR = pathlib.Path("/content/behaviour-microscope")

if not REPO_DIR.exists():
    !git clone --depth 1 $REPO_URL $REPO_DIR
else:
    if subprocess.run(["git", "-C", str(REPO_DIR), "status", "--porcelain"],
                      capture_output=True, text=True).stdout.strip():
        !git -C $REPO_DIR stash -u
    !git -C $REPO_DIR fetch --depth 1 origin main -q
    !git -C $REPO_DIR reset --hard origin/main -q

os.chdir(REPO_DIR)
if str(REPO_DIR / "src") not in sys.path:
    sys.path.insert(0, str(REPO_DIR / "src"))
!git -C $REPO_DIR log --oneline -1

## 2. Install


In [ ]:
%pip install -q -e '/content/behaviour-microscope'
print("installed")

## 3. GPU


In [ ]:
import torch

NEEDED_GB = 30
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> GPU."
p = torch.cuda.get_device_properties(0)
total = p.total_memory / 1e9
print(f"{p.name}  |  {total:.0f} GB  |  compute {p.major}.{p.minor}")
assert total > NEEDED_GB + 4, f"Needs ~{NEEDED_GB} GB of weights plus headroom."

DTYPE = "bfloat16" if torch.cuda.is_bf16_supported() else "float16"
DEVICE = "cuda"
print("dtype:", DTYPE, " device:", DEVICE)

## 4. Config


In [ ]:
from microscope.experiment import RunConfig, run_all
from microscope.scenarios import ARMS

MODEL_ID = "Qwen/Qwen3-14B"
ENABLE_THINKING = True
MAX_GEN_TOKENS = 2048

cfg = RunConfig(
    model_id=MODEL_ID, provider="local", backend="eager", dtype=DTYPE,
    extra_load_kwargs={"device": DEVICE},
    enable_thinking=ENABLE_THINKING,
    n_candidate_layers=4,
    max_gen_tokens=MAX_GEN_TOKENS,
)

for arm in ARMS:
    print(f"  {arm.name:20s} {arm.cue or '(no assertion)'}")
print()
print("model:", MODEL_ID, "| reasoning:", ENABLE_THINKING)


## 5. Run

Worker thread: interp-engine's sync facade cannot run inside Colab's kernel event loop.


In [ ]:
import concurrent.futures

with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
    run_dir = pool.submit(run_all, cfg).result()
run_dir


## 6. Quality report

`quality_report.json` is computed from this run. `FAIL` means the rates below are not usable as findings (`src/microscope/quality.py`). Unparsed responses are dropped from every rate; raise `MAX_GEN_TOKENS` and re-run rather than interpreting a failed parse.


In [ ]:
report = json.loads((run_dir / "quality_report.json").read_text())
print("Overall:", report["overall"].upper(), "\n")
for c in report["checks"]:
    print(f"[{c['status'].upper():4s}] {c['name']}: {c['detail']}")

In [ ]:
s = json.loads((run_dir / "summary.json").read_text())["behavioural"]
print(f"{'arm':22s} {'accepts false':>14s} {'accuracy':>9s} {'n scored':>9s}")
for arm in ["floor","junior_said","junior_confirmed","partner_said",
            "partner_confirmed","court","adverse"]:
    if arm not in s["fpar_by_arm"]: continue
    print(f"  {arm:20s} {s['fpar_by_arm'][arm]:13.0%} "
          f"{s['accuracy_by_arm'].get(arm, float('nan')):8.0%} "
          f"{s['n_scored_by_arm'][arm]:8d}/30")
print(f"\nparse failures: {s['parse_failures']}/{s['n_measurements']}")

## 7. Export


In [ ]:
import shutil
archive = shutil.make_archive(f"/content/{run_dir.name}", "zip", run_dir)
print(archive)
try:
    from google.colab import files
    files.download(archive)
except Exception as exc:
    print(f"Download from the file browser instead ({exc})")